# House sales

RealAgents is a real estate company that focuses on selling houses.

RealAgents sells a variety of types of house in one metropolitan area.

Some houses sell slowly and sometimes require lowering the price in order to find a buyer.

In order to stay competitive, RealAgents would like to optimize the listing prices of the houses it is trying to sell.

They want to do this by predicting the sale price of a house given its characteristics.

If they can predict the sale price in advance, they can decrease the time to sale.


## Data

The dataset contains records of previous houses sold in the area.

| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
| house_id    | Nominal. </br> Unique identifier for houses. </br>Missing values not possible. |
| city        | Nominal. </br>The city in which the house is located. One of 'Silvertown', 'Riverford', 'Teasdale' and 'Poppleton'. </br>Replace missing values with "Unknown". |
| sale_price  | Discrete. </br>The sale price of the house in whole dollars. Values can be any positive number greater than or equal to zero.</br>Remove missing entries. |
| sale_date   | Discrete. </br>The date of the last sale of the house. </br>Replace missing values with 2023-01-01. |
| months_listed  | Continuous. </br>The number of months the house was listed on the market prior to its last sale, rounded to one decimal place. </br>Replace missing values with mean number of months listed, to one decimal place. |
| bedrooms    | Discrete. </br>The number of bedrooms in the house. Any positive values greater than or equal to zero. </br>Replace missing values with the mean number of bedrooms, rounded to the nearest integer. |
| house_type   | Ordinal. </br>One of "Terraced" (two shared walls), "Semi-detached" (one shared wall), or "Detached" (no shared walls). </br>Replace missing values with the most common house type. |
| area      | Continuous. </br>The area of the house in square meters, rounded to one decimal place. </br>Replace missing values with the mean, to one decimal place. |



In [5]:
import pandas as pd
import numpy as np

# Load all three datasets
house_sales_df = pd.read_csv('house_sales.csv')
train_df       = pd.read_csv('train.csv')
validation_df  = pd.read_csv('validation.csv')

print("Datasets loaded successfully.")
print(f"  house_sales : {house_sales_df.shape}")
print(f"  train       : {train_df.shape}")
print(f"  validation  : {validation_df.shape}")


def clean_house_sales_data(df):
    """
    Apply the full data cleaning pipeline to a house sales dataframe.

    Handles:
      - city       : replaces '--' placeholder and fills NaN with 'Unknown'
      - sale_price : drops rows where the target is missing
      - sale_date  : converts to datetime, fills NaN with 2023-01-01
      - months_listed : converts to numeric, fills NaN with column mean
      - bedrooms   : converts to numeric, fills NaN with rounded mean
      - house_type : standardises abbreviations (Det., Semi, Terr.) to full names,
                     fills NaN with the most common type
      - area       : strips the ' sq.m.' suffix if present, converts to numeric,
                     fills NaN with column mean

    Parameters
    ----------
    df : pd.DataFrame
        Raw house sales dataframe (e.g. house_sales.csv or train.csv).

    Returns
    -------
    pd.DataFrame
        Cleaned dataframe with no missing values.
    """
    clean_df = df.copy()

    # city: '--' is used as a placeholder for missing values
    clean_df['city'] = clean_df['city'].replace('--', np.nan).fillna('Unknown')

    # sale_price: we cannot impute the target variable, so we drop missing rows
    clean_df = clean_df.dropna(subset=['sale_price'])

    # sale_date: parse to datetime so we can do date arithmetic later if needed
    clean_df['sale_date'] = pd.to_datetime(clean_df['sale_date'], errors='coerce')
    clean_df['sale_date'] = clean_df['sale_date'].fillna(pd.Timestamp('2023-01-01'))

    # months_listed: fill missing with the mean, rounded to 1 decimal place
    clean_df['months_listed'] = pd.to_numeric(clean_df['months_listed'], errors='coerce')
    mean_months = round(clean_df['months_listed'].mean(), 1)
    clean_df['months_listed'] = clean_df['months_listed'].fillna(mean_months)

    # bedrooms: fill missing with the mean, rounded to the nearest whole number
    clean_df['bedrooms'] = pd.to_numeric(clean_df['bedrooms'], errors='coerce')
    mean_bedrooms = round(clean_df['bedrooms'].mean())
    clean_df['bedrooms'] = clean_df['bedrooms'].fillna(mean_bedrooms).astype(int)

    # house_type: some rows use abbreviations instead of the full category names
    clean_df['house_type'] = clean_df['house_type'].replace({
        'Det.' : 'Detached',
        'Semi' : 'Semi-detached',
        'Terr.': 'Terraced'
    })
    most_common_type = clean_df['house_type'].mode()[0]
    clean_df['house_type'] = clean_df['house_type'].fillna(most_common_type)

    # area: the raw file appends ' sq.m.' to each value as a string suffix
    clean_df['area'] = clean_df['area'].astype(str).str.replace(' sq.m.', '', regex=False)
    clean_df['area'] = pd.to_numeric(clean_df['area'], errors='coerce')
    mean_area = round(clean_df['area'].mean(), 1)
    clean_df['area'] = clean_df['area'].fillna(mean_area)

    return clean_df


# Apply cleaning to the full house_sales dataset and verify the result
clean_data = clean_house_sales_data(house_sales_df)

print("\nMissing values after cleaning (should all be 0):")
print(clean_data.isna().sum())

print(f"\nUnique house_type values: {clean_data['house_type'].unique()}")
print(f"Shape after cleaning: {clean_data.shape}")

clean_data.head()

Datasets loaded successfully.
  house_sales : (1500, 8)
  train       : (1200, 8)
  validation  : (300, 7)

Missing values after cleaning (should all be 0):
house_id         0
city             0
sale_price       0
sale_date        0
months_listed    0
bedrooms         0
house_type       0
area             0
dtype: int64

Unique house_type values: ['Semi-detached' 'Detached' 'Terraced']
Shape after cleaning: (1500, 8)


,house_id,city,sale_price,sale_date,months_listed,bedrooms,house_type,area
0,1217792,Silvertown,55943,2021-09-12,5.4,2,Semi-detached,107.8
1,1900913,Silvertown,384677,2021-01-17,6.3,5,Detached,498.8
2,1174927,Riverford,281707,2021-11-10,6.9,6,Detached,542.5
3,1773666,Silvertown,373251,2020-04-13,6.1,6,Detached,528.4
4,1258487,Silvertown,328885,2020-09-24,8.7,5,Detached,477.1


# Task #1 Ridge Regression (L2 Regularization)

Ridge regression adds an **L2 penalty** to the OLS objective, stabilising the coefficient estimates when features are correlated (multicollinearity):

$$L_{\text{Ridge}}(\beta) = \|y - X\beta\|_2^2 + \lambda \|\beta\|_2^2$$

Closed-form solution:
$$\hat\beta_{\text{Ridge}} = (X^\top X + \lambda I)^{-1} X^\top y$$

- **`RidgeCV`** automatically finds the optimal $\lambda$ via 5-fold cross-validation.
- All coefficients are **shrunk toward zero** but never reach exactly zero.
- Saves the trained model to `streamlitApp/models/` for the Streamlit app.

In [6]:

# Ridge Regression with Cross-Validated Lambda

import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge, RidgeCV, LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Define which columns are numeric and which are categorical
numeric_features     = ['bedrooms', 'area', 'months_listed']
categorical_features = ['city', 'house_type']
all_features         = numeric_features + categorical_features
target               = 'sale_price'

# Clean and prepare the training data
clean_train_df = clean_house_sales_data(train_df)
X_full = clean_train_df[all_features].copy()
y_full = clean_train_df[target].copy()

# Build the preprocessing pipeline:
#   - StandardScaler  : normalises numeric features to mean=0, std=1
#   - OneHotEncoder   : converts each category into binary dummy columns
try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    # sklearn < 1.2 uses the old parameter name 'sparse'
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocessor_reg = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', ohe,              categorical_features),
])

X_processed = preprocessor_reg.fit_transform(X_full)

# Retrieve the full list of feature names after one-hot encoding
cat_names     = preprocessor_reg.named_transformers_['cat'] \
                                .get_feature_names_out(categorical_features).tolist()
feature_names = numeric_features + cat_names

print(f"Total features after encoding: {len(feature_names)}")
print(f"Feature list: {feature_names}\n")

# Split 80% for training and 20% for testing
X_tr, X_te, y_tr, y_te = train_test_split(
    X_processed, y_full.values, test_size=0.2, random_state=42
)

# Fit an OLS model as a baseline — we will compare Ridge against this
ols_model = LinearRegression()
ols_model.fit(X_tr, y_tr)
ols_pred  = ols_model.predict(X_te)
ols_rmse  = np.sqrt(mean_squared_error(y_te, ols_pred))
print(f"OLS (all features)  RMSE on test: ${ols_rmse:,.2f}")

# Use RidgeCV to search for the best lambda through 5-fold cross-validation.
# Ridge needs a larger lambda range than Lasso because it operates on eigenvalues
# of X'X, which can be in the range of tens to hundreds.
ridge_alphas = [0.01, 0.1, 1, 5, 10, 50, 100, 500, 1000, 5000]
ridge_cv     = RidgeCV(alphas=ridge_alphas, cv=5)
ridge_cv.fit(X_tr, y_tr)
best_ridge_alpha = ridge_cv.alpha_
print(f"\nBest Ridge lambda (5-fold CV): {best_ridge_alpha}")

# Refit Ridge with the CV-optimal lambda
ridge_model = Ridge(alpha=best_ridge_alpha)
ridge_model.fit(X_tr, y_tr)
ridge_pred  = ridge_model.predict(X_te)
ridge_rmse  = np.sqrt(mean_squared_error(y_te, ridge_pred))
print(f"Ridge (lambda={best_ridge_alpha})  RMSE on test: ${ridge_rmse:,.2f}")

# Generate predictions on the held-out validation set
X_val_processed = preprocessor_reg.transform(validation_df[all_features])
ridge_val_pred  = ridge_model.predict(X_val_processed)
ridge_result    = pd.DataFrame({
    'house_id': validation_df['house_id'],
    'price':    ridge_val_pred,
})
print("\nRidge — validation predictions (first 5 rows):")
display(ridge_result.head())

# Compare OLS and Ridge coefficients.
# Ridge shrinks all coefficients toward zero but never eliminates any of them.
ridge_coef_df = pd.DataFrame({
    'feature':    feature_names,
    'ols_coef':   ols_model.coef_,
    'ridge_coef': ridge_model.coef_,
}).sort_values('ridge_coef', key=abs, ascending=False)

print(f"\nTop 10 Coefficients — OLS vs Ridge (lambda={best_ridge_alpha}):")
display(ridge_coef_df.head(10))
print(f"\nRidge keeps all features — zero coefficients: {(ridge_model.coef_ == 0).sum()} / {len(ridge_model.coef_)}")


Total features after encoding: 10
Feature list: ['bedrooms', 'area', 'months_listed', 'city_Poppleton', 'city_Riverford', 'city_Silvertown', 'city_Teasdale', 'house_type_Detached', 'house_type_Semi-detached', 'house_type_Terraced']

OLS (all features)  RMSE on test: $23,465.56

Best Ridge lambda (5-fold CV): 0.1
Ridge (lambda=0.1)  RMSE on test: $23,469.61

Ridge — validation predictions (first 5 rows):


,house_id,price
0,1331375,121135.238591
1,1630115,302796.017761
2,1645745,384074.103790
3,1336775,124281.782450
4,1888274,270696.143392



Top 10 Coefficients — OLS vs Ridge (lambda=0.1):


,feature,ols_coef,ridge_coef
1,area,89748.269315,89535.478543
4,city_Riverford,-48697.161172,-48663.555204
6,city_Teasdale,48572.165964,48546.503261
9,house_type_Terraced,-33326.918572,-33311.255643
7,house_type_Detached,32601.770872,32601.105242
5,city_Silvertown,17842.681464,17833.006673
3,city_Poppleton,-17717.686256,-17715.954731
0,bedrooms,5495.710842,5702.686013
2,months_listed,1285.620426,1283.508725
8,house_type_Semi-detached,725.147699,710.150401



Ridge keeps all features — zero coefficients: 0 / 10


# Task #2 Lasso Regression (L1 Regularization / Feature Selection)

Lasso replaces the squared penalty with an **absolute-value (L1) penalty**:

$$L_{\text{Lasso}}(\beta) = \|y - X\beta\|_2^2 + \lambda \|\beta\|_1$$

**Why Lasso creates exact zeros, the threshold condition:**

$$\hat\beta_j = 0 \quad \text{if and only if} \quad |2 X_j^\top (y - X\hat\beta)| \leq \lambda$$

The `|β|` function has a **corner at zero** (subgradient = interval [−1, 1]), allowing coefficients to "stick" at exactly zero — something Ridge's smooth `β²` penalty can never do.

- **`LassoCV`** finds the optimal $\lambda$ via 5-fold cross-validation.
- Irrelevant features are **automatically eliminated** (coefficient = exactly 0).
- Different $\lambda$ scale than Ridge: Lasso operates on gradient magnitudes ≈ 0.01–10.

In [7]:

# Lasso Regression with Cross-Validated Lambda

from sklearn.linear_model import Lasso, LassoCV

# Use LassoCV to find the best lambda through 5-fold cross-validation.
# Lasso uses a much smaller lambda range than Ridge because it operates on
# gradient magnitudes (|2 * Xj' * residual|), which are typically 0.01 to 5
# after standardisation — so lambdas in that same range directly compete.
lasso_alphas = [0.001, 0.005, 0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0, 5.0, 10.0]
lasso_cv     = LassoCV(alphas=lasso_alphas, cv=5, max_iter=50000, random_state=42)
lasso_cv.fit(X_tr, y_tr)
best_lasso_alpha = lasso_cv.alpha_
print(f"Best Lasso lambda (5-fold CV): {best_lasso_alpha}")

# Fit the final Lasso model using the CV-optimal lambda
lasso_model = Lasso(alpha=best_lasso_alpha, max_iter=50000)
lasso_model.fit(X_tr, y_tr)
lasso_pred  = lasso_model.predict(X_te)
lasso_rmse  = np.sqrt(mean_squared_error(y_te, lasso_pred))
print(f"Lasso (lambda={best_lasso_alpha})  RMSE on test: ${lasso_rmse:,.2f}")

# Count how many features Lasso kept and how many it eliminated.
# A coefficient of exactly 0 means Lasso decided that feature is not useful.
n_selected   = int((lasso_model.coef_ != 0).sum())
n_eliminated = int((lasso_model.coef_ == 0).sum())

print(f"\nLasso feature selection (lambda = {best_lasso_alpha}):")
print(f"  Features selected  (non-zero coef) : {n_selected}")
print(f"  Features eliminated (coef = 0)     : {n_eliminated}")

# Build a side-by-side comparison table of all three models
lasso_status  = ['selected' if c != 0 else 'eliminated (= 0)' for c in lasso_model.coef_]
lasso_coef_df = pd.DataFrame({
    'feature':      feature_names,
    'ols_coef':     ols_model.coef_.round(4),
    'ridge_coef':   ridge_model.coef_.round(4),
    'lasso_coef':   lasso_model.coef_.round(4),
    'lasso_status': lasso_status,
}).sort_values('lasso_coef', key=abs, ascending=False)

print("\nAll Coefficients — OLS vs Ridge vs Lasso:")
display(lasso_coef_df)

# A feature is zeroed out when its correlation with the residuals is below the
# threshold: |2 * Xj' * (y - X*beta)| <= lambda.
# Ridge's smooth beta^2 penalty never achieves this — its gradient at zero is 0,
# not an interval — so Ridge only shrinks, never eliminates.
print(f"\nLasso threshold condition (lambda = {best_lasso_alpha}):")
print(f"  Feature j is zeroed out when |2 * Xj' * residual| <= {best_lasso_alpha}")
print(f"  {n_eliminated} feature(s) did not exceed this threshold and were eliminated.")

# Generate predictions on the validation set
lasso_val_pred = lasso_model.predict(X_val_processed)
lasso_result   = pd.DataFrame({
    'house_id': validation_df['house_id'],
    'price':    lasso_val_pred,
})
print("\nLasso — validation predictions (first 5 rows):")
display(lasso_result.head())


Best Lasso lambda (5-fold CV): 2.0
Lasso (lambda=2.0)  RMSE on test: $23,465.14

Lasso feature selection (lambda = 2.0):
  Features selected  (non-zero coef) : 9
  Features eliminated (coef = 0)     : 1

All Coefficients — OLS vs Ridge vs Lasso:


,feature,ols_coef,ridge_coef,lasso_coef,lasso_status
1,area,89748.2693,89535.4785,89748.9984,selected
4,city_Riverford,-48697.1612,-48663.5552,-66518.5383,selected
3,city_Poppleton,-17717.6863,-17715.9547,-35545.8628,selected
9,house_type_Terraced,-33326.9186,-33311.2556,-34030.3841,selected
7,house_type_Detached,32601.7709,32601.1052,31868.1756,selected
6,city_Teasdale,48572.1660,48546.5033,30726.5490,selected
0,bedrooms,5495.7108,5702.6860,5497.8910,selected
2,months_listed,1285.6204,1283.5087,1283.2085,selected
5,city_Silvertown,17842.6815,17833.0067,0.0201,selected
8,house_type_Semi-detached,725.1477,710.1504,0.0000,eliminated (= 0)



Lasso threshold condition (lambda = 2.0):
  Feature j is zeroed out when |2 * Xj' * residual| <= 2.0
  1 feature(s) did not exceed this threshold and were eliminated.

Lasso — validation predictions (first 5 rows):


,house_id,price
0,1331375,121089.978397
1,1630115,302860.665018
2,1645745,384082.574127
3,1336775,124230.225924
4,1888274,270745.802824


# Model Comparison and Save Models

Compare all models on the same test split, then **save** the trained artifacts to `streamlitApp/models/` so the interactive Streamlit app can load them without retraining.

**Saved artifacts:**
| File | Contents |
|------|----------|
| `preprocessor.joblib` | Fitted `ColumnTransformer` (StandardScaler + OHE) |
| `ols_model.joblib` | Fitted `LinearRegression` (all features) |
| `ridge_model.joblib` | Fitted `Ridge` with CV-optimal λ |
| `lasso_model.joblib` | Fitted `Lasso` with CV-optimal λ |
| `model_metadata.json` | Alphas, feature names, RMSE scores, training info |

In [8]:

# Comparison created model and and Save, compare all models on the same test split,
# save the trained models to StreamlitApp

import joblib
import json
import os
from datetime import date

# Fit an OLS model on bedrooms only as an additional baseline.
# Column 0 of X_tr corresponds to the scaled 'bedrooms' feature.
ols_bedrooms = LinearRegression()
ols_bedrooms.fit(X_tr[:, 0:1], y_tr)
bed_pred = ols_bedrooms.predict(X_te[:, 0:1])
bed_rmse = np.sqrt(mean_squared_error(y_te, bed_pred))

# Summarise all models on the same 20% test split for a fair comparison
comparison_df = pd.DataFrame({
    'model': [
        'OLS (bedrooms only)',
        'OLS (all features)',
        f'Ridge  (lambda = {best_ridge_alpha})',
        f'Lasso  (lambda = {best_lasso_alpha})',
    ],
    'test_rmse': [
        round(bed_rmse, 2),
        round(ols_rmse, 2),
        round(ridge_rmse, 2),
        round(lasso_rmse, 2),
    ],
    'features_used': [
        '1 (bedrooms only)',
        f'{len(feature_names)} (all)',
        f'{len(feature_names)} (all, shrunk)',
        f'{n_selected} / {len(feature_names)} (selected)',
    ],
    'regularization': ['None', 'None', 'L2 (Ridge)', 'L1 (Lasso)'],
})

print("Model Comparison — same 20% test split:")
display(comparison_df)

# Re-fit all models on the full training data before saving.
# The train/test split above was only for evaluation — production models
# should learn from as many samples as possible.
print("\nRe-fitting all models on the full training data.......")
preprocessor_reg.fit(X_full)
X_all = preprocessor_reg.transform(X_full)

ols_final   = LinearRegression().fit(X_all, y_full)
ridge_final = Ridge(alpha=best_ridge_alpha).fit(X_all, y_full)
lasso_final = Lasso(alpha=best_lasso_alpha, max_iter=50000).fit(X_all, y_full)

print("All models re-fitted on the full training data.")

# Save the trained artifacts so the Streamlit app can load them without retraining.
# The preprocessor must be saved too — it holds the fitted scaler and encoder
# parameters and must be applied to any new input before prediction.
save_dir = os.path.join('streamlitApp', 'models')
os.makedirs(save_dir, exist_ok=True)

joblib.dump(preprocessor_reg, os.path.join(save_dir, 'preprocessor.joblib'))
joblib.dump(ols_final,        os.path.join(save_dir, 'ols_model.joblib'))
joblib.dump(ridge_final,      os.path.join(save_dir, 'ridge_model.joblib'))
joblib.dump(lasso_final,      os.path.join(save_dir, 'lasso_model.joblib'))

# Save a JSON metadata file with the key parameters and evaluation scores.
# The Streamlit app reads this to display results without re-running CV.
metadata = {
    'ridge_alpha':           float(best_ridge_alpha),
    'lasso_alpha':           float(best_lasso_alpha),
    'feature_names':         feature_names,
    'numeric_features':      numeric_features,
    'categorical_features':  categorical_features,
    'n_features':            len(feature_names),
    'n_train_samples':       int(len(X_full)),
    'test_rmse_ols':         round(float(ols_rmse),   2),
    'test_rmse_ridge':       round(float(ridge_rmse), 2),
    'test_rmse_lasso':       round(float(lasso_rmse), 2),
    'lasso_n_selected':      int(n_selected),
    'lasso_n_eliminated':    int(n_eliminated),
    'training_date':         str(date.today()),
}
with open(os.path.join(save_dir, 'model_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\nSaved to: {os.path.abspath(save_dir)}/")
for fname in ['preprocessor.joblib', 'ols_model.joblib',
              'ridge_model.joblib',   'lasso_model.joblib',
              'model_metadata.json']:
    fpath = os.path.join(save_dir, fname)
    size  = os.path.getsize(fpath)
    print(f"  {fname:<30}  {size / 1024:.1f} KB")

print(f"\nRidge CV-optimal lambda : {best_ridge_alpha}")
print(f"Lasso CV-optimal lambda : {best_lasso_alpha}  ({n_selected}/{len(feature_names)} features selected)")


Model Comparison — same 20% test split:


,model,test_rmse,features_used,regularization
0,OLS (bedrooms only),47805.80,1 (bedrooms only),None
1,OLS (all features),23465.56,10 (all),None
2,Ridge (lambda = 0.1),23469.61,"10 (all, shrunk)",L2 (Ridge)
3,Lasso (lambda = 2.0),23465.14,9 / 10 (selected),L1 (Lasso)



Re-fitting all models on the full training data.......
All models re-fitted on the full training data.

Saved to: C:\Users\HP\Desktop\Blossom_Academy\house-sales-price-prediction\streamlitApp\models/
  preprocessor.joblib             3.1 KB
  ols_model.joblib                0.7 KB
  ridge_model.joblib              0.6 KB
  lasso_model.joblib              0.7 KB
  model_metadata.json             0.7 KB

Ridge CV-optimal lambda : 0.1
Lasso CV-optimal lambda : 2.0  (9/10 features selected)
